# ScribbleAI — end-to-end Colab runbook

Run top to bottom on a **T4 GPU** runtime (*Runtime → Change runtime type → T4 GPU*).
Everything that must survive a disconnect (processed data, checkpoints, results) lives on Google Drive
under `MyDrive/scribbleai/`. If Colab disconnects, reconnect and re-run from **step 1**; training resumes automatically.

**One-time preparation on Drive** (do this from your laptop in the Drive web UI):
```
MyDrive/scribbleai/raw/ujipenchars2.txt        <- from the UCI zip
MyDrive/scribbleai/raw/uji2.names
MyDrive/scribbleai/raw/corpus.txt              <- optional: plain English text (e.g. a Project Gutenberg book)
MyDrive/scribbleai/inhouse/*.jsonl             <- collector exports, one file per writer
```

## 1. Clone the repo, install, check the environment

In [ ]:
import getpass, os
REPO_USER = "<github-user>"      # <- change
if not os.path.exists("/content/scribbleai"):
    token = getpass.getpass("GitHub token (hidden): ")
    !git clone https://{token}@github.com/{REPO_USER}/scribbleai.git /content/scribbleai
    del token
%cd /content/scribbleai
!git pull -q
!pip install -q -r requirements.txt
!pip install -q --no-deps pyctcdecode
!python setup_env.py

## 2. Mount Drive and link data folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/scribbleai"
CKPT = f"{DRIVE}/checkpoints"
!mkdir -p {DRIVE}/raw {DRIVE}/inhouse {DRIVE}/processed {CKPT} {DRIVE}/results
!mkdir -p data/raw/ujipenchars2
!cp -n {DRIVE}/raw/ujipenchars2.txt {DRIVE}/raw/uji2.names data/raw/ujipenchars2/ 2>/dev/null; ls -la data/raw/ujipenchars2
!cp -n {DRIVE}/raw/corpus.txt data/raw/ 2>/dev/null; true
!cp -n {DRIVE}/inhouse/*.jsonl data/inhouse/ 2>/dev/null; ls data/inhouse

## 3. Unit tests (all must pass; the real-UJI tests run now that the file is present)

In [ ]:
!python -m pytest tests -q

## 4. Look at the data before training on it
If the UJI characters are upside down, set `data.uji_flip_y: true` in `configs/base.yaml`.

In [ ]:
!python src/preprocessing/parse_uji.py
!python -m src.preprocessing.visualize uji --n 10
!python -m src.preprocessing.visualize synth --n 5
from IPython.display import Image, display
display(Image("results/figures/uji_samples.png")); display(Image("results/figures/synth_samples.png"))

In [ ]:
# in-house data (only once you have collected some)
import glob
if glob.glob("data/inhouse/*.jsonl"):
    !python -m src.preprocessing.visualize inhouse --n 5
    display(Image("results/figures/inhouse_samples.png"))

## 5. Build the dataset (vocab, synthetic val/test lines, feature stats, in-house sets)
Takes a few minutes. The result is copied to Drive so you only rebuild when preprocessing or the in-house data changes.

In [ ]:
!python -m src.preprocessing.build_dataset
!grep -E "vocab_size|blank_index" configs/base.yaml
!cp -r data/processed/. {DRIVE}/processed/

In [ ]:
# after a disconnect, restore instead of rebuilding:
# !mkdir -p data/processed && cp -r {DRIVE}/processed/. data/processed/

## 6. Baseline classifier (Objective 2)

In [ ]:
!python -m src.models.baseline && cp results/baseline.json {DRIVE}/results/

## 7. Sanity checks — do not train until ALL CHECKS PASSED

In [ ]:
!python -m src.sanity_checks --steps 600

## 8. Train (checkpoints go straight to Drive; re-running this cell resumes)

In [ ]:
import wandb; wandb.login()
!python -m src.train --ckpt-dir {CKPT} --wandb

## 9. Language model + spell checker resources

In [ ]:
!python -m src.decoding.lm && cp data/processed/char_lm.json data/processed/word_3gram.arpa {DRIVE}/processed/

## 10. Evaluate (all decoders, latency, confusion matrix, error analysis, generalisation gap)

In [ ]:
!python -m src.evaluate --ckpt {CKPT}/best.pt --max-items 2000
!cp -r results/. {DRIVE}/results/
from IPython.display import Markdown
import glob
display(Markdown(open(sorted(glob.glob("results/*best/summary.md"))[-1]).read()))

## 11. (Optional) In-house fine-tuning experiment
Needs `data.inhouse_test_writers` set in the config **before** building the dataset, so that `inhouse_train` exists.

In [ ]:
!python -m src.train --finetune --init {CKPT}/best.pt --ckpt-dir {CKPT}_finetune
!python -m src.evaluate --ckpt {CKPT}_finetune/best.pt --sets inhouse_test test --out results/finetune
!cp -r results/. {DRIVE}/results/

## 12. Get the model onto the demo laptop
Download `MyDrive/scribbleai/checkpoints/best.pt` → `scribbleai/checkpoints/best.pt` and
`MyDrive/scribbleai/processed/char_lm.json` → `scribbleai/data/processed/char_lm.json` on the laptop, then:
```
python demo/app.py --ckpt checkpoints/best.pt
```